# ResNet50 Fine-tuned Model
### Chest X-Ray Classification: Normal vs Pneumonia vs COVID-19
This notebook implements full fine-tuning of ResNet50 by unfreezing the last few layers and adding data augmentation to improve on the feature extraction baseline.

In [1]:
# Core PyTorch libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

/opt/anaconda3/envs/pneumonia-detection/lib/python3.11/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/opt/anaconda3/envs/pneumonia-detection/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <EB3FF92A-5EB1-3EE8-AF8B-5923C1265422> /opt/anaconda3/envs/pneumonia-detection/lib/python3.11/site-packages/torchvision/image.so
  Reason: tried: '/opt/anaconda3/envs/pneumonia-detection/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/anaconda3/envs/pneumonia-detection/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/anaconda3/envs/pneumonia-detection/lib/python3.11/lib-dynload/../../libjpeg.9.dylib' (no such file), '/opt/anaconda3/envs/pneumonia-detection/bin/../lib/libjpeg.9.dylib' (no such file)'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warnin

In [2]:
# Use Apple GPU if available, otherwise CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# Dataset paths
data_dir = Path('../data/Dataset')
train_dir = data_dir / 'Train_Validation'
test_dir = data_dir / 'Test'
classes = ['COVID', 'Normal', 'Pneumonia']

Using device: mps


In [3]:
# Training transform WITH data augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),        # randomly flip image horizontally
    transforms.RandomRotation(10),             # randomly rotate up to 10 degrees
    transforms.ColorJitter(brightness=0.2),   # randomly adjust brightness
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Validation/test transform WITHOUT augmentation
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

In [4]:
class XRayDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        for label, cls in enumerate(classes):
            class_dir = self.data_dir / cls
            for img_path in class_dir.glob('*'):
                self.image_paths.append(img_path)
                self.labels.append(label)
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, index):
        img_path = self.image_paths[index]
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)
        label = self.labels[index]
        return image, label

# Create datasets with different transforms
full_dataset = XRayDataset(train_dir, transform=train_transform)
test_dataset = XRayDataset(test_dir, transform=val_transform)

# Split FIRST before training
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_split, val_split = random_split(full_dataset, [train_size, val_size])

# Override validation transform
val_split.dataset.transform = val_transform

# Create DataLoaders
train_loader = DataLoader(train_split, batch_size=32, shuffle=True)
val_loader = DataLoader(val_split, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Training samples:", train_size)
print("Validation samples:", val_size)
print("Test samples:", len(test_dataset))

Training samples: 2308
Validation samples: 578
Test samples: 341


In [5]:
# Load pretrained ResNet50
model = models.resnet50(weights='IMAGENET1K_V1')

# Freeze all layers first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the last two residual blocks (layer3 and layer4)
for param in model.layer3.parameters():
    param.requires_grad = True

for param in model.layer4.parameters():
    param.requires_grad = True

# Replace final layer with 3-class classifier
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 3)

# Move to device
model = model.to(device)

print("ResNet50 loaded with fine-tuning setup")
print(f"Final layer: {model.fc}")

ResNet50 loaded with fine-tuning setup
Final layer: Linear(in_features=2048, out_features=3, bias=True)


In [6]:
# Different learning rates for different parts of the model
# Lower lr for pretrained layers, higher lr for new layers
optimizer = optim.Adam([
    {'params': model.layer3.parameters(), 'lr': 0.0001},  # pretrained - small lr
    {'params': model.layer4.parameters(), 'lr': 0.0001},  # pretrained - small lr
    {'params': model.fc.parameters(), 'lr': 0.001}         # new layer - larger lr
])

criterion = nn.CrossEntropyLoss()

print("Optimiser configured with differential learning rates")

Optimiser configured with differential learning rates


In [7]:
# Different learning rates for different parts of the model
# Lower lr for pretrained layers, higher lr for new layers
optimizer = optim.Adam([
    {'params': model.layer3.parameters(), 'lr': 0.0001},  # pretrained - small lr
    {'params': model.layer4.parameters(), 'lr': 0.0001},  # pretrained - small lr
    {'params': model.fc.parameters(), 'lr': 0.001}         # new layer - larger lr
])

criterion = nn.CrossEntropyLoss()

print("Optimiser configured with differential learning rates")

Optimiser configured with differential learning rates


In [8]:
num_epochs = 10

for epoch in range(num_epochs):
    # Training phase
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    
    # Validation phase
    model.eval()
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    val_acc = 100 * val_correct / val_total
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% - Val Acc: {val_acc:.2f}%")

Epoch 1/10 - Loss: 0.3248 - Train Acc: 87.22% - Val Acc: 91.87%
Epoch 2/10 - Loss: 0.0988 - Train Acc: 97.49% - Val Acc: 93.08%
Epoch 3/10 - Loss: 0.0849 - Train Acc: 97.79% - Val Acc: 89.79%
Epoch 4/10 - Loss: 0.0865 - Train Acc: 97.49% - Val Acc: 91.35%
Epoch 5/10 - Loss: 0.0374 - Train Acc: 98.61% - Val Acc: 94.12%
Epoch 6/10 - Loss: 0.0489 - Train Acc: 99.31% - Val Acc: 93.25%
Epoch 7/10 - Loss: 0.0645 - Train Acc: 98.01% - Val Acc: 92.39%
Epoch 8/10 - Loss: 0.0762 - Train Acc: 97.62% - Val Acc: 93.08%
Epoch 9/10 - Loss: 0.0487 - Train Acc: 98.44% - Val Acc: 93.43%
Epoch 10/10 - Loss: 0.0550 - Train Acc: 99.18% - Val Acc: 93.43%
